<center><h1>Steven-Schneider_Isla_HW3</h1></center>
<br>
<br>

Name: Isla Steven-Schneider
<br>
Github Username: islastevenschneider
<br>
USC ID: 1985-4980-09

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [1]:
import pandas as pd
import numpy as np
import os

Get the AReM Data Set

In [2]:
root = os.path.join("..", "data", "AReM")
cols = ["time", "avg_rss12", "var_rss12", "avg_rss13", "var_rss13", "avg_rss23", "var_rss23"]
data = {}

for folder_name in os.listdir(root):
    folder_path = os.path.join(root, folder_name)
    if not os.path.isdir(folder_path):
        continue
    for file in os.listdir(folder_path):
        if not file.endswith(".csv"):
            continue
        num = int(file.replace("dataset", "").replace(".csv", ""))
        data[(folder_name, num)] = pd.read_csv(os.path.join(folder_path, file), skiprows=5, names=cols, usecols=range(7), sep=r"[,\s]+", engine="python")

### (b) Test and Train Data

In [3]:
test_set = {}
train_set = {}

for (folder_name, num), df in data.items():
    if folder_name.startswith("bending"):
        max_test = 2
    else:
        max_test = 3
    if num <= max_test:
        test_set[(folder_name, num)] = df
    else:
        train_set[(folder_name, num)] = df

In [4]:
len(test_set), len(train_set)

(19, 69)

### (c) Feature Extraction

#### i. Research

Features used in time series classification:

1) Energy: root mean square (RMS), energy, peak-to-peak amplitude
2) Extremes: minimum, maximum, first and third quartiles, percentiles
3) Location: mean, median, mode
4) Over time: number of peaks, autocorrelation, slope, mean-crossing rate
5) Shape: skewness, kurtosis
6) Spread: standard deviation, variance, range, interquartile range, mean absolute deviation

These features are standard in common time series feature extraction libraries, including TSFEL (Barandas et al., 2020) and tsfresh (Christ et al., 2018).

#### ii. Extraction

In [5]:
rows = []

for (folder_name, num), df in data.items():
    row = {"activity": folder_name, "dataset": num}
    for i, col in enumerate(cols[1:], start=1):
        series = df[col]
        row[f"min{i}"] = series.min()
        row[f"max{i}"] = series.max()
        row[f"mean{i}"] = series.mean()
        row[f"median{i}"] = series.median()
        row[f"std{i}"] = series.std()
        row[f"1st_quart{i}"] = series.quantile(0.25)
        row[f"3rd_quart{i}"] = series.quantile(0.75)
    rows.append(row)

features = pd.DataFrame(rows)
features = features.sort_values(["activity", "dataset"]).reset_index(drop=True)
features

,activity,dataset,min1,max1,mean1,median1,std1,1st_quart1,3rd_quart1,min2,...,std5,1st_quart5,3rd_quart5,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6
0,bending1,1,37.25,45.00,40.624792,40.50,1.476967,39.25,42.0000,0.0,...,2.188449,33.0000,36.00,0.00,1.92,0.570583,0.43,0.582915,0.0000,1.3000
1,bending1,2,38.00,45.67,42.812812,42.50,1.435550,42.00,43.6700,0.0,...,1.995255,32.0000,34.50,0.00,3.11,0.571083,0.43,0.601010,0.0000,1.3000
2,bending1,3,35.00,47.40,43.954500,44.33,1.558835,43.00,45.0000,0.0,...,1.999604,35.3625,36.50,0.00,1.79,0.493292,0.43,0.513506,0.0000,0.9400
3,bending1,4,33.00,47.75,42.179812,43.50,3.670666,39.15,45.0000,0.0,...,3.849448,30.4575,36.33,0.00,2.18,0.613521,0.50,0.524317,0.0000,1.0000
4,bending1,5,33.00,45.75,41.678063,41.75,2.243490,41.33,42.7500,0.0,...,2.411026,28.4575,31.25,0.00,1.79,0.383292,0.43,0.389164,0.0000,0.5000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,walking,11,19.50,45.33,33.586875,34.25,4.650935,30.25,37.0000,0.0,...,3.283983,13.7300,18.25,0.00,8.32,3.259729,3.11,1.640243,2.0500,4.3225
84,walking,12,19.75,45.50,34.322750,35.25,4.752477,31.00,38.0000,0.0,...,3.119856,13.5000,17.75,0.00,9.67,3.432563,3.20,1.732727,2.1575,4.5650
85,walking,13,19.50,46.00,34.546229,35.25,4.842294,31.25,37.8125,0.0,...,2.823124,14.0000,17.75,0.00,10.00,3.338125,3.08,1.656742,2.1600,4.3350
86,walking,14,23.50,46.25,34.873229,35.25,4.531720,31.75,38.2500,0.0,...,3.131076,13.7500,18.00,0.00,9.51,3.424646,3.27,1.690960,2.1700,4.5000


#### iii. Standard Deviation

In [6]:
np.random.seed(1009)
n_boot = 1000

ci_rows = []

for col in features.columns[2:]:
    values = features[col].values
    boot_stds = []
    for b in range(n_boot):
        sample = np.random.choice(values, size=len(values), replace=True)
        boot_stds.append(sample.std(ddof=1))
    lower, upper = np.percentile(boot_stds, [5, 95])
    ci_rows.append({"feature": col, "std": values.std(ddof=1), "ci_lower": lower, "ci_upper": upper})

ci_table = pd.DataFrame(ci_rows)
ci_table

,feature,std,ci_lower,ci_upper
0,min1,9.569975,8.202351,10.806503
1,max1,4.394362,3.373652,5.344513
2,mean1,5.335718,4.704152,5.883542
3,median1,5.440054,4.794615,5.997379
4,std1,1.772153,1.568022,1.956938
5,1st_quart1,6.153590,5.580884,6.655082
6,3rd_quart1,5.138925,4.290066,5.847510
7,min2,0.000000,0.000000,0.000000
8,max2,5.062729,4.613099,5.407962
9,mean2,1.574164,1.400514,1.708003


The std column shows how much each feature varies across the 88 instances, and the CI columns give a 90% bootstrap confidence interval for that standard deviation. min2 and min4 have a standard deviation and interval of exactly 0, since every instance has a minimum of 0 on those series. The features from the signal strength series have the widest intervals (for example min5, from about 4.4 to 7.5), while the features from the series that measure how much the signal fluctuates have much narrower ones.

#### iv. Select Features

In [7]:
pd.set_option("display.max_columns", None)
features.drop(columns="dataset").groupby("activity").mean()

,min1,max1,mean1,median1,std1,1st_quart1,3rd_quart1,min2,max2,mean2,median2,std2,1st_quart2,3rd_quart2,min3,max3,mean3,median3,std3,1st_quart3,3rd_quart3,min4,max4,mean4,median4,std4,1st_quart4,3rd_quart4,min5,max5,mean5,median5,std5,1st_quart5,3rd_quart5,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6
activity,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
bending1,35.642857,46.795714,42.667723,42.904286,1.912853,41.505714,44.012857,0.0,1.875714,0.454360,0.472857,0.366494,0.122857,0.618571,4.464286,28.618571,19.158830,19.517857,3.839324,17.000000,21.743571,0.0,6.314286,0.793399,0.532857,0.890122,0.184286,1.082143,20.988571,36.524286,30.201982,30.595714,2.607278,28.861071,31.761429,0.000000,2.715714,0.552390,0.458571,0.531610,0.061429,0.962857
bending2,17.541667,44.958333,29.415344,28.993333,4.537836,26.354167,31.847083,0.0,6.385000,0.447622,0.358333,0.661918,0.000000,0.583750,1.583333,27.360000,15.326375,15.791667,4.982278,12.364583,18.698333,0.0,6.850000,0.755219,0.485000,0.903203,0.071667,0.980000,4.723333,32.721667,18.728476,18.305833,4.870860,15.695000,21.562500,0.000000,7.231667,0.981906,0.773333,0.905411,0.468333,1.168333
cycling,24.377333,44.483333,36.747143,36.221667,3.772555,34.063833,39.871833,0.0,10.849333,2.688332,2.199667,1.892871,1.343333,3.732000,5.733333,26.389333,16.396611,16.356000,3.470477,14.043333,18.725500,0.0,9.218667,2.989278,2.758333,1.704133,1.675667,4.017500,5.484000,24.488667,17.398582,17.619333,3.025294,15.488000,19.617000,0.000000,9.175333,3.037336,2.715333,1.749433,1.720500,4.037833
lying,33.216667,42.733333,39.850217,39.278000,2.024855,39.044000,41.644667,0.0,2.444667,0.343333,0.313333,0.286817,0.184000,0.446000,0.066667,14.715333,7.264876,7.800000,3.043814,5.111333,9.249333,0.0,4.196000,0.580935,0.496000,0.610658,0.172000,0.786500,0.466667,17.594667,9.258135,9.405333,3.598072,6.711667,11.718833,0.000000,4.040667,0.660632,0.540000,0.560337,0.340833,0.896667
sitting,35.544667,46.356000,41.748634,41.359000,2.051710,40.172000,43.527333,0.0,3.277333,0.401416,0.346000,0.370950,0.177333,0.592667,1.044667,23.531333,14.864351,15.538667,4.619691,12.086000,18.335000,0.0,5.925333,0.750700,0.542000,0.838328,0.143333,0.993000,2.216667,25.200000,16.101094,16.452667,4.436935,13.214333,19.647333,0.000000,6.094000,0.902887,0.781333,0.821134,0.445333,1.145000
standing,34.744000,47.120000,43.857747,44.412000,2.189060,42.785000,45.510000,0.0,2.900667,0.398604,0.374000,0.367512,0.086000,0.500333,2.366667,21.316667,14.773876,15.197667,3.163898,13.471000,16.875500,0.0,5.584667,0.616150,0.458333,0.709681,0.086000,0.811167,1.233333,21.078000,13.704479,14.056000,3.430426,11.896167,16.068667,0.000000,5.313333,0.913015,0.801333,0.710320,0.463333,1.165667
walking,18.816000,46.261333,34.433676,35.069333,4.788857,31.207167,37.858333,0.0,14.000667,4.337393,3.959667,2.439142,2.525000,5.799833,6.267333,23.572000,15.394560,15.414333,2.908019,13.458500,17.359500,0.0,9.054667,3.203608,3.007667,1.621026,2.026667,4.221833,6.350000,24.183333,16.025058,16.138667,3.099009,14.060333,18.200000,0.028667,9.126667,3.343612,3.119000,1.699922,2.105833,4.408167


I believe the three most important time-domain features are mean, standard deviation, and max. The odd series (1, 3, and 5) measure signal strength between pairs of sensors, which depends on body position, while the even series (2, 4, and 6) measure how much that signal fluctuates, which reflects movement.

The mean captures both position and movement. For position, mean3 is 7.3 for lying versus 15 to 19 for every other activity, so a low mean3 points to lying. For movement, mean2 is 4.3 for walking and 2.7 for cycling, compared to about 0.4 for the stiller activities, so a high mean2 suggests an activity involving movement.

Standard deviation is important because it captures how much the signal changes during a recording. In std1, walking, cycling, and bending2 sit around 4 to 5, while lying, sitting, standing, and bending1 are around 2. Walking and cycling involve continuous movement, and bending2 is a crouching, twisting reach rather than bending1’s standing forward bend, so these activities produce a less stable signal than ones where the body holds a steadier position.

Max is important because movement causes spikes in the signal. In max2, cycling (10.8) and walking (14.0) are well above the other activities (1.9 to 6.4).

## 2. ISLR 3.7.4

### (a) Linear Train

I would expect the training RSS for the cubic regression to be slightly lower than for the linear regression. Since the cubic model can reproduce the linear model by setting β₂ and β₃ equal to zero, least squares will never give it a higher training RSS, so it must be less than or equal to the linear RSS. In practice it will be slightly lower, because the extra terms fit some of the random noise in the training data, even though the true relationship is linear.

### (b) Linear Test

I would expect the test RSS for the linear regression to be lower than for the cubic regression. Since the true relationship is linear, the extra terms in the cubic model do not capture any real structure in the data; they only fit the noise in the training data. This noise does not carry over to new test data, so the cubic model will do slightly worse on the test set, giving it a higher test RSS. In other words, the cubic model overfits: both models are unbiased, but the cubic has higher variance.

### (c) Not Linear Train

I would expect the training RSS to be lower for the cubic regression. As established in (a), the cubic model contains the linear model as a special case (β₂ = β₃ = 0), so least squares can never give it a higher training RSS than the linear model, regardless of the true relationship. Since the true relationship is not linear, the extra β₂ and β₃ terms can also capture some of the real curvature in the data, so the cubic’s training RSS should be noticeably lower.

### (d) Not Linear Testing

There is not enough information to tell. If the true relationship is far from linear, the cubic model’s extra terms will learn the real pattern and apply it well to new data, so it would have a lower test RSS. If the relationship is only slightly nonlinear, the linear model is already close to the truth. In that case the cubic’s extra terms mostly fit noise, as in (b), and the linear model could do better on the test set. Since we don’t know how far from linear the relationship is, we can’t say which test RSS will be lower.

## 3. ISLR 3.7.3 - Extra Practice 

## 3. ISLR 3.7.5 - Extra Practice 